# 02 — Exploratory Data Analysis

**Goal.** Understand how patient volume varies across time, weather, and calendar drivers before we engineer features or fit models. Every chart here is designed either to validate a domain assumption or to earn a place in the final narrative.

**Questions this notebook answers:**
1. What does the time series look like — level, seasonality, shocks?
2. Which day-of-week and month-level patterns dominate?
3. How do weather, holidays, and flu/hay-fever seasons shift volume?
4. How does volume stack when multiple illness drivers co-occur?
5. What autocorrelation structure should lag features target?
6. Is the target skewed enough to justify a log transform?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_palette('deep')

df = pd.read_csv(Path.cwd().parent / 'data' / 'raw' / 'clinic_patients_melbourne.csv', parse_dates=['date'])
print(f'{len(df)} rows · {df.date.min().date()} → {df.date.max().date()}')
df.head()

## 1. Summary statistics

In [ ]:
df[['patients', 'temperature', 'precipitation', 'humidity', 'pollen_index']].describe().round(1)

## 2. Time series — level, seasonality, shocks

In [ ]:
df_ts = df.set_index('date')
ma30 = df_ts['patients'].rolling(30, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_ts.index, df_ts['patients'], color='grey', alpha=0.5, linewidth=0.8, label='Daily')
ax.plot(ma30.index, ma30, color='#c0392b', linewidth=2, label='30-day MA')

shock = df_ts[df_ts['is_thunderstorm_asthma'] == 1]
ax.scatter(shock.index, shock['patients'], color='black', s=70, zorder=5, label='Thunderstorm asthma')

ax.set_title('Daily patient volume — 3-year view')
ax.set_ylabel('Patients / day')
ax.legend()
plt.tight_layout(); plt.show()

## 3. Day-of-week and monthly patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_mean = df.groupby('day_name')['patients'].mean().reindex(dow_order)
colors = ['#c0392b' if d == 'Monday' else '#3498db' for d in dow_order]
axes[0, 0].bar(dow_order, dow_mean.values, color=colors)
axes[0, 0].axhline(df['patients'].mean(), color='black', linestyle='--', alpha=0.5, label='Overall mean')
axes[0, 0].set_title('Day of week — Monday dominates (weekend backlog + GP gap)')
axes[0, 0].set_ylabel('Avg patients')
axes[0, 0].tick_params(axis='x', rotation=30)
axes[0, 0].legend()

monthly = df.groupby('month')['patients'].mean()
axes[0, 1].bar(monthly.index, monthly.values,
              color=['#c0392b' if m in (6, 7, 8) else '#3498db' for m in monthly.index])
axes[0, 1].axhline(df['patients'].mean(), color='black', linestyle='--', alpha=0.5)
axes[0, 1].set_title('Month — winter flu peak (Jun–Aug)')
axes[0, 1].set_xlabel('Month'); axes[0, 1].set_ylabel('Avg patients')
axes[0, 1].set_xticks(range(1, 13))

quarterly = df.groupby('quarter')['patients'].mean()
axes[1, 0].bar(quarterly.index.astype(str), quarterly.values, color='#2c3e50')
axes[1, 0].set_title('Quarter')
axes[1, 0].set_ylabel('Avg patients')

pivot = df.pivot_table(index='day_name', columns='month', values='patients', aggfunc='mean').reindex(dow_order)
sns.heatmap(pivot, cmap='RdYlBu_r', ax=axes[1, 1], cbar_kws={'label': 'Avg patients'})
axes[1, 1].set_title('Day-of-week × month heatmap')

plt.tight_layout(); plt.show()

## 4. Weather and temperature effects

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

wt_order = ['Sunny', 'Partly Cloudy', 'Cloudy', 'Rainy']
wt_mean = df.groupby('weather_type')['patients'].mean().reindex(wt_order)
axes[0].bar(wt_order, wt_mean.values,
            color=['#f1c40f', '#95a5a6', '#7f8c8d', '#34495e'])
axes[0].axhline(df['patients'].mean(), color='black', linestyle='--', alpha=0.5)
axes[0].set_title('Weather — rainy/cloudy lift respiratory visits')
axes[0].set_ylabel('Avg patients')

axes[1].scatter(df['temperature'], df['patients'], alpha=0.3, s=12, color='#2c3e50')
bins = pd.cut(df['temperature'], bins=12)
binned = df.groupby(bins, observed=True)['patients'].mean()
axes[1].plot([i.mid for i in binned.index], binned.values, color='#c0392b', linewidth=2, label='Binned mean')
axes[1].set_xlabel('Temperature (°C)'); axes[1].set_ylabel('Patients')
axes[1].set_title('Temperature — U-shape: extremes drive volume')
axes[1].legend()

axes[2].scatter(df['pollen_index'], df['patients'], alpha=0.4, s=14, color='#27ae60')
pi_mean = df.groupby('pollen_index')['patients'].mean()
axes[2].plot(pi_mean.index, pi_mean.values, color='#c0392b', linewidth=2)
axes[2].set_xlabel('Pollen index'); axes[2].set_ylabel('Patients')
axes[2].set_title('Pollen — continuous hay-fever driver')

plt.tight_layout(); plt.show()

## 5. Calendar and epidemiological drivers

Holiday effect is **positive** here — GPs close and walk-ins absorb overflow. Opposite sign vs. leisure venues.

In [ ]:
factors = [
    ('is_public_holiday', 'Public holiday'),
    ('is_day_after_public_holiday', 'Day after public holiday'),
    ('is_school_holiday', 'School holiday'),
    ('is_flu_peak', 'Flu peak weeks'),
    ('is_hayfever_season', 'Hay fever season'),
    ('temp_extreme_hot', 'Extreme heat (>35°C)'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (col, label) in zip(axes.flat, factors):
    means = df.groupby(col)['patients'].mean()
    vals = [means.get(0, np.nan), means.get(1, np.nan)]
    ax.bar(['Off', 'On'], vals, color=['#95a5a6', '#c0392b'])
    if not np.isnan(vals[0]) and not np.isnan(vals[1]):
        uplift = (vals[1] - vals[0]) / vals[0] * 100
        ax.set_title(f'{label}  (+{uplift:.0f}%)')
    else:
        ax.set_title(label)
    ax.set_ylabel('Avg patients')
plt.tight_layout(); plt.show()

## 6. Stacking effect — illness driver count

In [ ]:
stack = df.groupby('illness_driver_count')['patients'].agg(['mean', 'count']).round(1)
stack.columns = ['Avg patients', 'Days']
print(stack)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(stack.index.astype(str), stack['Avg patients'], color='#2c3e50')
ax.axhline(df['patients'].mean(), color='#c0392b', linestyle='--', label='Overall mean')
ax.set_xlabel('Number of concurrent illness drivers')
ax.set_ylabel('Avg patients')
ax.set_title('Stacking: each additional driver adds volume')
ax.legend()
plt.tight_layout(); plt.show()

## 7. Autocorrelation — which lags matter?

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
plot_acf(df['patients'], lags=60, ax=axes[0], title='ACF — watch for spikes at 7, 14, 21 (weekly cycle)')
plot_pacf(df['patients'], lags=60, ax=axes[1], method='ywm', title='PACF — independent signal at lag 1 and lag 7')
plt.tight_layout(); plt.show()

## 8. Log transform check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['patients'], bins=40, color='#2c3e50', alpha=0.85, edgecolor='white')
axes[0].set_title(f"Raw (skew={stats.skew(df['patients']):.2f})")
axes[0].set_xlabel('Patients')

log_y = np.log1p(df['patients'])
axes[1].hist(log_y, bins=40, color='#27ae60', alpha=0.85, edgecolor='white')
axes[1].set_title(f"log1p(patients) (skew={stats.skew(log_y):.2f})")
axes[1].set_xlabel('log1p(patients)')
plt.tight_layout(); plt.show()

## 9. Correlations with target

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()['patients'].drop('patients').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 8))
corr.plot(kind='barh', ax=ax, color=['#27ae60' if v > 0 else '#c0392b' for v in corr.values])
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('Correlation with patients (raw features only — lag/rolling come next notebook)')
ax.set_xlabel('Pearson r')
plt.tight_layout(); plt.show()

## Takeaways for feature engineering

- **Cyclical encoding** for day-of-week and month (Monday↔Sunday, Dec↔Jan should be adjacent).
- **Lag features** at 1, 7, 14, 28 days (weekly cycle is the strongest lag signal in ACF).
- **Rolling stats** (mean / std / max / min) at multiple windows — demand shifts gradually after seasonal transitions.
- **Interaction features** — public holiday × Monday, flu peak × rainy, pollen × hayfever season.
- **Log-transform target** — right-skew drops near normal after `log1p`.